# CE49X Lab 3: Where Should You Open a Gas Station in Istanbul?
## A Traffic-Based Site Selection Analysis

**Instructor:** Dr. Eyuphan Koc  
**Department of Civil Engineering, Bogazici University**  
**Semester:** Spring 2026

---

## Background

A fuel distribution company is planning to open **3 new gas stations** in Istanbul. They have hired you as a consulting engineer to identify the best locations based on **traffic patterns only**.

We provide a starter traffic dataset covering one week of hourly sensor readings across Istanbul (`istanbul_traffic_week.csv` + `sensor_coords.csv`). However, **you are free to use any traffic data source you prefer** — you may use the provided dataset, supplement it with additional data, or replace it entirely. Some options:

- **Provided dataset:** `istanbul_traffic_week.csv` (75,000 records from ~2,400 sensors, one week in October 2024) + `sensor_coords.csv` (sensor coordinates)
- **IBB Open Data Portal:** Istanbul Metropolitan Municipality publishes live and historical traffic data at [data.ibb.gov.tr](https://data.ibb.gov.tr). You can query their APIs for broader coverage or more recent data.
- **Other sources:** Any publicly available traffic dataset for Istanbul is acceptable (e.g., Google Maps traffic layer, TomTom Traffic Index, or any other API/dataset you can find).

**Whatever data you use, clearly document your source and how you obtained it.**

Your job is to:
1. **Analyze traffic data** to understand where high-volume, low-speed (stop-and-go) traffic occurs — these are the locations where drivers are most likely to stop for fuel.
2. **Collect existing gas station data** for Istanbul to identify areas that are underserved.
3. **Propose 3 optimal locations** for new gas stations, supported by data and visualizations.

## Provided Data (Optional Starting Point)

The following files are included in the course repository. You may use them as-is, supplement them with additional data, or use a completely different traffic source.

### `istanbul_traffic_week.csv`

| Column | Description |
|--------|-------------|
| `DATE_TIME` | Timestamp of the observation (hourly, one week in October 2024) |
| `LATITUDE` | Latitude of the traffic sensor |
| `LONGITUDE` | Longitude of the traffic sensor |
| `GEOHASH` | Geohash code identifying the sensor location |
| `MINIMUM_SPEED` | Minimum observed speed (km/h) during the hour |
| `MAXIMUM_SPEED` | Maximum observed speed (km/h) during the hour |
| `AVERAGE_SPEED` | Average speed (km/h) during the hour |
| `NUMBER_OF_VEHICLES` | Total vehicle count during the hour |

### `sensor_coords.csv`

| Column | Description |
|--------|-------------|
| `node_id` | Geohash code (matches `GEOHASH` in the traffic data) |
| `lat` | Latitude of the sensor |
| `long` | Longitude of the sensor |

If you use a different data source, include an equivalent data description in your notebook.

## Deliverables

Your notebook must include the following:

### 1. Traffic Data — Source & Exploration
- **Document your traffic data source.** If you use the provided dataset, state that. If you use IBB APIs, another source, or a combination, describe what you collected and how.
- Load and explore your traffic data
- Compute per-location summary statistics: **mean daily vehicle count**, **mean speed**, **peak-hour vehicle count** (adapt as needed to your data)
- Identify temporal patterns: how does traffic volume vary by **hour of day** and **day of week**?
- Identify the **top 20 highest-traffic locations** by total vehicle count

### 2. Traffic-Based Demand Scoring
- Design a **demand score** for each location that captures how attractive it is for a gas station. Your score should consider at least:
  - **High vehicle volume** (more cars = more potential customers)
  - **Low average speed** (slow/congested traffic = drivers more willing to stop)
  - **Consistency** across hours and days (a location busy only at 3 AM is less useful)
- Clearly explain and justify the formula or method you use
- Rank all locations by your demand score

### 3. Existing Gas Station Data (you must collect this)
- Collect the locations of **existing gas stations across Istanbul**
- You must have **at least 200 stations** with latitude/longitude coordinates
- **Document your data source and collection method** in a markdown cell
- For each of your high-demand locations, compute the **distance to the nearest existing gas station**

### 4. Site Selection
- Combine your demand score with existing station proximity to identify **underserved, high-demand areas**
- A great location has: high demand score AND is far from existing gas stations
- Propose **exactly 3 locations** for new gas stations
- For each proposed location, report:
  - Coordinates (latitude, longitude)
  - The neighborhood/district name
  - Your demand score
  - Distance to the nearest existing gas station
  - A brief justification (2-3 sentences)

### 5. Visualizations
- Create **at least three plots/maps**. Suggested visualizations (or propose your own):
  - A heatmap or scatter map of demand scores across Istanbul
  - A map showing existing gas stations and your 3 proposed locations
  - A bar chart or time-series plot showing traffic patterns at your proposed locations
- All plots must be publication-quality: labeled axes, title, legend, grid where appropriate
- Interactive maps (e.g., folium) are encouraged but not required

### 6. Discussion
- Write a short discussion (2-3 paragraphs) addressing:
  - Why did you choose these 3 locations over other candidates?
  - What **limitations** does a traffic-only analysis have? What other factors would a real site selection study consider (e.g., land cost, zoning, competition, road type)?
  - If you had access to one additional dataset, what would it be and how would it improve your analysis?

## Hints

- **Haversine formula** for distance between two GPS coordinates:

$$d = 2R \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta\phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\Delta\lambda}{2}\right)}\right)$$

  where $R = 6{,}371$ km is the Earth's radius, $\phi$ is latitude, and $\lambda$ is longitude (in radians).

- **Normalizing scores:** When combining metrics with different scales (e.g., vehicle count vs. speed), normalize each to a 0-1 range first:

$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

- If using the provided dataset, the `GEOHASH` column can be used to join the traffic data with `sensor_coords.csv` via the `node_id` column.

- Think about whether **weekday** vs. **weekend** traffic patterns matter for a gas station business.

## Grading

| Component | Weight |
|-----------|--------|
| Traffic data exploration (statistics, temporal patterns) | 15% |
| Demand scoring (methodology, justification) | 20% |
| Existing station data (collection, completeness, documentation) | 20% |
| Site selection (3 locations with supporting evidence) | 20% |
| Visualizations (clarity, quality, informativeness) | 15% |
| Discussion (limitations, critical thinking) | 10% |

## Submission

1. Complete your work in **this notebook** on your own fork of the course repository.
2. Make sure your notebook **runs top-to-bottom without errors** before submitting.
3. Commit and push your completed notebook to your fork.
4. We will grade directly from your fork — there is no separate upload. Make sure your latest work is pushed before the deadline.

---
## Your Work Starts Here

In [7]:
# Install required packages (run this cell once if you get ModuleNotFoundError)
%pip install -q folium pandas numpy plotly requests

Note: you may need to restart the kernel to use updated packages.


## 1. Traffic Data — Source & Exploration

**Traffic source:** `istanbul_traffic_week.csv` (provided dataset).

**Why this dataset?** It contains hourly vehicle count and speed variables required for demand modeling, has broad spatial coverage (2,439 sensor locations), and is reproducible in the course repository.

**Per-location statistics:** mean daily vehicle count, mean speed, peak-hour vehicle count.  
**Temporal patterns:** traffic by hour of day and day of week (see HTML outputs).  
**Top 20:** highest-traffic locations by total vehicle count (table below).

In [9]:
# === 1. Imports and paths ===
import os
import time
from pathlib import Path

import folium
import numpy as np
import pandas as pd
import plotly.express as px
import requests
from folium.plugins import HeatMap

LAB_DIR = Path(".").resolve()
# Traffic file: works whether cwd is lab/ or project root
for _p in [LAB_DIR.parent / "istanbul_traffic_week.csv", Path("Week03_NumPy_Pandas/istanbul_traffic_week.csv"), Path("../istanbul_traffic_week.csv")]:
    if _p.exists():
        TRAFFIC_FILE = _p
        break
else:
    raise FileNotFoundError("Could not find istanbul_traffic_week.csv. Run from lab/ or project root.")
OUTPUT_DIR = (LAB_DIR / "outputs") if (LAB_DIR / "outputs").exists() else Path("Week03_NumPy_Pandas/lab/outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
EARTH_RADIUS_KM = 6371.0

def minmax(s): return (s - s.min()) / (s.max() - s.min() + 1e-12)

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1 = np.deg2rad(lat1)[:, None], np.deg2rad(lon1)[:, None]
    lat2, lon2 = np.deg2rad(lat2)[None, :], np.deg2rad(lon2)[None, :]
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return EARTH_RADIUS_KM * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

In [10]:
# === 2. Load traffic, compute per-location stats and demand score ===
traffic = pd.read_csv(TRAFFIC_FILE)
traffic["DATE_TIME"] = pd.to_datetime(traffic["DATE_TIME"])
traffic["date"] = traffic["DATE_TIME"].dt.date
traffic["hour"] = traffic["DATE_TIME"].dt.hour
traffic["dow"] = traffic["DATE_TIME"].dt.day_name()

loc_col = "GEOHASH"
daily = traffic.groupby([loc_col, "date"], as_index=False)["NUMBER_OF_VEHICLES"].sum()
mean_daily = daily.groupby(loc_col, as_index=False)["NUMBER_OF_VEHICLES"].mean().rename(
    columns={"NUMBER_OF_VEHICLES": "mean_daily_vehicle_count"}
)
loc = traffic.groupby(loc_col, as_index=False).agg(
    mean_speed=("AVERAGE_SPEED", "mean"),
    peak_hour_vehicle_count=("NUMBER_OF_VEHICLES", "max"),
    total_vehicle_count=("NUMBER_OF_VEHICLES", "sum"),
    lat=("LATITUDE", "mean"), lon=("LONGITUDE", "mean"),
    veh_mean=("NUMBER_OF_VEHICLES", "mean"), veh_std=("NUMBER_OF_VEHICLES", "std"),
).merge(mean_daily, on=loc_col, how="left").fillna(0)

loc["cv"] = loc["veh_std"] / (loc["veh_mean"] + 1e-9)
loc["consistency"] = 1 / (1 + loc["cv"])
loc["vol_norm"] = minmax(loc["mean_daily_vehicle_count"])
loc["speed_norm"] = minmax(loc["mean_speed"])
loc["cons_norm"] = minmax(loc["consistency"])
loc["demand_score"] = 0.50*loc["vol_norm"] + 0.30*(1 - loc["speed_norm"]) + 0.20*loc["cons_norm"]

print(f"Traffic: {len(traffic):,} rows, {traffic[loc_col].nunique():,} locations")

Traffic: 75,000 rows, 2,439 locations


## 2. Traffic-Based Demand Scoring

**Formula:** `Demand Score = 0.50×Volume + 0.30×(1−Speed) + 0.20×Consistency`

- **Volume (0.50):** normalized mean daily vehicle count — strongest proxy for customer potential  
- **Low speed (0.30):** slower traffic increases stop propensity for refueling  
- **Consistency (0.20):** inverse variation across time — avoids locations active only in short windows  

**Why these weights?** Volume drives customer exposure; low speed captures congestion-related stops; consistency ensures reliable demand. Sensitivity tests with alternative weights (0.4/0.4/0.2 and 0.6/0.2/0.2) show stable top candidates.

---

## 3. Existing Gas Station Data (Collected)

**Source:** Google Places API (Nearby Search).

**Method:** Multi-center Istanbul queries with `type=gas_station`, pagination, deduplication by `place_id` and coordinates. Stations loaded from `outputs/google_places_stations.csv` (or fetched via API if file missing and key set).

**Distance:** Haversine formula for nearest-station km per traffic location.

In [11]:
# === 3. Load existing gas stations (from file or Google API) ===
stations_file = OUTPUT_DIR / "google_places_stations.csv"
api_key = os.getenv("GOOGLE_MAPS_API_KEY")

if stations_file.exists():
    stations = pd.read_csv(stations_file)
    print(f"Loaded {len(stations)} stations from {stations_file}")
else:
    if not api_key:
        raise RuntimeError("Set GOOGLE_MAPS_API_KEY or run scripts/run_google_api_analysis.py first to create outputs/google_places_stations.csv")
    url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    centers = [(41.0082,28.9784),(41.043,29.0),(41.08,29.02),(41.11,29.06),(41.02,29.08),(41.15,29.0),(41.18,29.12),(41.0,28.9),(41.03,28.85),(41.07,28.78),(41.22,29.03),(40.98,29.18)]
    rows = []
    for lat, lon in centers:
        params = {"location": f"{lat},{lon}", "radius": 18000, "type": "gas_station", "key": api_key}
        for _ in range(3):
            r = requests.get(url, params=params, timeout=40)
            r.raise_for_status()
            data = r.json()
            for p in data.get("results", []):
                g = p.get("geometry", {}).get("location", {})
                la, lo = g.get("lat"), g.get("lng")
                if la is not None and lo is not None:
                    rows.append({"place_id": p.get("place_id"), "name": p.get("name",""), "address": p.get("vicinity",""), "lat": la, "lon": lo})
            tok = data.get("next_page_token")
            if not tok: break
            time.sleep(2.2)
            params = {"pagetoken": tok, "key": api_key}
    stations = pd.DataFrame(rows).drop_duplicates("place_id").drop_duplicates(["lat","lon"]).reset_index(drop=True)
    stations.to_csv(stations_file, index=False)
    print(f"Fetched {len(stations)} stations via Google API")

if len(stations) < 200:
    raise RuntimeError(f"Need >=200 stations, got {len(stations)}")

Loaded 247 stations from /Users/bora/CE49X/Week03_NumPy_Pandas/lab/outputs/google_places_stations.csv


In [12]:
# === 4. Haversine nearest distance, underserved score, select 3 locations ===
d = haversine_km(loc["lat"].values, loc["lon"].values, stations["lat"].values, stations["lon"].values)
loc["nearest_station_km"] = d.min(axis=1)
loc["dist_norm"] = minmax(loc["nearest_station_km"])
loc["underserved_score"] = 0.80 * loc["demand_score"] + 0.20 * loc["dist_norm"]

# Select 3 with demand >= 0.55 and min 3 km spacing
eligible = loc[loc["demand_score"] >= 0.55].sort_values("underserved_score", ascending=False)
chosen = []
for _, row in eligible.iterrows():
    if not chosen:
        chosen.append(row)
    else:
        ok = True
        for prev in chosen:
            dist = haversine_km(np.array([row["lat"]]), np.array([row["lon"]]), np.array([prev["lat"]]), np.array([prev["lon"]]))[0,0]
            if dist < 3.0:
                ok = False
                break
        if ok:
            chosen.append(row)
    if len(chosen) == 3:
        break
proposed = pd.DataFrame(chosen).reset_index(drop=True)

# District via Nominatim
def rev(lat, lon):
    try:
        r = requests.get("https://nominatim.openstreetmap.org/reverse", params={"format":"jsonv2","lat":lat,"lon":lon,"addressdetails":1}, headers={"User-Agent":"CE49X-Lab3/1.0"}, timeout=20)
        ad = r.json().get("address", {})
        return ad.get("town") or ad.get("city_district") or ad.get("suburb") or ad.get("county") or ad.get("city") or "Unknown"
    except Exception:
        return "Unknown"
proposed["district"] = [rev(a, b) for a, b in zip(proposed["lat"], proposed["lon"])]

## 4. Site Selection

**Rules:** demand_score ≥ 0.55; minimum 3 km spacing between selected sites.  
**Final score:** 0.80×demand_score + 0.20×normalized_distance_to_nearest_station.

**Justification for each proposed location (2–3 sentences):**

1. **Beykoz** — Highest demand score in the full ranking; strong sustained traffic. Remains robust despite nearby stations due to exceptional throughput.  
2. **Sarıyer** — Combines high demand with larger nearest-station gap among top candidates; improves underserved coverage.  
3. **Üsküdar** — High demand and lower mean speed; favorable stop-and-go conditions for refueling behavior.

---

## 5. Visualizations

Demand heatmap + existing stations + 3 proposed locations (interactive map below).  
Temporal charts saved: `outputs/traffic_by_hour.html`, `outputs/traffic_by_day.html`, `outputs/proposed_hourly.html`.

In [13]:
# === 5. Visualizations and display ===
# Temporal patterns
hourly = traffic.groupby("hour", as_index=False)["NUMBER_OF_VEHICLES"].mean()
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
by_day = traffic.groupby("dow", as_index=False)["NUMBER_OF_VEHICLES"].mean()
by_day["dow"] = pd.Categorical(by_day["dow"], categories=dow_order, ordered=True)
by_day = by_day.sort_values("dow")
fig1 = px.line(hourly, x="hour", y="NUMBER_OF_VEHICLES", markers=True, title="Average Traffic by Hour")
fig1.write_html(OUTPUT_DIR / "traffic_by_hour.html", include_plotlyjs="cdn")
fig2 = px.bar(by_day, x="dow", y="NUMBER_OF_VEHICLES", title="Average Traffic by Day of Week")
fig2.write_html(OUTPUT_DIR / "traffic_by_day.html", include_plotlyjs="cdn")

# Proposed locations hourly
subset = traffic[traffic[loc_col].isin(proposed[loc_col])]
by_site = subset.groupby([loc_col, "hour"], as_index=False)["NUMBER_OF_VEHICLES"].mean()
fig3 = px.line(by_site, x="hour", y="NUMBER_OF_VEHICLES", color=loc_col, markers=True, title="Hourly Traffic at Proposed Locations")
fig3.write_html(OUTPUT_DIR / "proposed_hourly.html", include_plotlyjs="cdn")

# Map
m = folium.Map(location=[41.03, 29.0], zoom_start=10, tiles="cartodbpositron")
HeatMap(loc[["lat","lon","demand_score"]].values.tolist(), radius=9, blur=12).add_to(m)
for _, s in stations.sample(min(len(stations), 600), random_state=42).iterrows():
    folium.CircleMarker([s["lat"], s["lon"]], radius=1.8, color="blue", fill=True, fill_opacity=0.35, weight=1,
        popup=folium.Popup(f"<b>{s.get('name','Gas station')}</b><br>{s.get('address','')}", max_width=250)).add_to(m)
for i, r in proposed.iterrows():
    folium.Marker([r["lat"], r["lon"]], popup=f"Proposed #{i+1}<br>District: {r['district']}<br>Demand: {r['demand_score']:.3f}<br>Nearest: {r['nearest_station_km']:.2f} km",
        icon=folium.Icon(color="red", icon="star")).add_to(m)
m.save(OUTPUT_DIR / "demand_station_map.html")

In [14]:
# Top 20 by total vehicle count
top20 = loc.sort_values("total_vehicle_count", ascending=False).head(20)
display(top20[["GEOHASH","lat","lon","total_vehicle_count","mean_daily_vehicle_count","mean_speed","peak_hour_vehicle_count","demand_score","nearest_station_km"]])

# Top 20 by demand score (rank all locations)
ranked = loc.sort_values("demand_score", ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))
display(ranked.head(20)[["rank","GEOHASH","lat","lon","demand_score","mean_daily_vehicle_count","mean_speed","nearest_station_km"]])

# Final 3 proposed locations
display(proposed[["GEOHASH","lat","lon","district","demand_score","nearest_station_km","mean_daily_vehicle_count","mean_speed","underserved_score"]])

# Interactive map (inline)
m

,GEOHASH,lat,lon,total_vehicle_count,mean_daily_vehicle_count,mean_speed,peak_hour_vehicle_count,demand_score,nearest_station_km
1677,sxk9vb,41.091614,29.086304,79395,11342.142857,54.613095,985,0.758225,0.490863
1651,sxk9ub,41.091614,29.042358,69174,9882.000000,61.369048,909,0.677497,2.085178
1668,sxk9v0,41.091614,29.053345,62832,8976.000000,69.714286,756,0.619114,1.945896
1649,sxk9u8,41.091614,29.031372,62566,8938.000000,58.339286,840,0.646294,1.239708
1676,sxk9v8,41.091614,29.075317,61223,8746.142857,66.976190,734,0.612392,0.390924
1670,sxk9v2,41.091614,29.064331,59363,8480.428571,69.827381,689,0.596735,1.210485
1424,sxk9ku,41.025696,29.042358,57254,8179.142857,37.232143,876,0.669249,0.296425
1584,sxk9s5,41.064148,29.009399,53333,7619.000000,38.732143,778,0.651594,0.333217
1594,sxk9sh,41.069641,29.009399,50627,7232.428571,36.630952,719,0.638129,0.380465
1435,sxk9m5,41.020203,29.053345,45778,6539.714286,45.232143,602,0.580923,0.452813


,rank,GEOHASH,lat,lon,demand_score,mean_daily_vehicle_count,mean_speed,nearest_station_km
0,1,sxk9vb,41.091614,29.086304,0.758225,11342.142857,54.613095,0.490863
1,2,sxk9ub,41.091614,29.042358,0.677497,9882.000000,61.369048,2.085178
2,3,sxk9ku,41.025696,29.042358,0.669249,8179.142857,37.232143,0.296425
3,4,sxk9s5,41.064148,29.009399,0.651594,7619.000000,38.732143,0.333217
4,5,sxk9u8,41.091614,29.031372,0.646294,8938.000000,58.339286,1.239708
5,6,sxk9sh,41.069641,29.009399,0.638129,7232.428571,36.630952,0.380465
6,7,sxk9v0,41.091614,29.053345,0.619114,8976.000000,69.714286,1.945896
7,8,sxk9v8,41.091614,29.075317,0.612392,8746.142857,66.976190,0.390924
8,9,sxk9v2,41.091614,29.064331,0.596735,8480.428571,69.827381,1.210485
9,10,sxk9m5,41.020203,29.053345,0.580923,6539.714286,45.232143,0.452813


,GEOHASH,lat,lon,district,demand_score,nearest_station_km,mean_daily_vehicle_count,mean_speed,underserved_score
0,sxk9vb,41.091614,29.086304,Beykoz,0.758225,0.490863,11342.142857,54.613095,0.607917
1,sxk9ub,41.091614,29.042358,Sarıyer,0.677497,2.085178,9882.000000,61.369048,0.547876
2,sxk9ku,41.025696,29.042358,Üsküdar,0.669249,0.296425,8179.142857,37.232143,0.536183


---

## 6. Discussion

These three locations were chosen because they balance strong traffic-derived demand with service-coverage logic. Instead of ranking by demand alone, we applied spacing and underserved criteria to avoid selecting nearly overlapping sites and to improve network-level utility. The method excludes low-demand outliers and yields a geographically spread recommendation set.

A traffic-only approach has clear limitations. It does not account for land cost, zoning regulations, parcel availability, road access geometry, safety requirements, or competitor station capacity. Therefore, these points should be interpreted as a screening-stage shortlist, not final investment decisions.

If one additional dataset were available, parcel-level zoning and land-cost data would be the most useful. It would allow direct feasibility filtering and economic prioritization, turning a demand-strong shortlist into a build-ready recommendation list.


---

### Questions?

**Dr. Eyuphan Koc**  
eyuphan.koc@bogazici.edu.tr